# Generating segmentation predictions

In [ ]:
from utils.device import get_device

device = get_device()

In [ ]:
from utils.available_datasets import available_datasets

dataset_choice = available_datasets["PERSEVERE_subset"]

train_split = dataset_choice.preferred_train_split
data_dir = dataset_choice.data_dir
splits_filepath = dataset_choice.splits_filepath
ndim = dataset_choice.ndim
input_channels = dataset_choice.input_channels

## Loading the model

### Case #1: Use the provided weights (see [README](../README.md)) to initialize the pretrained U-Net model

In [ ]:
from image_segmentation.models import BinarySegmentator

provided_ckpt_path = dataset_choice.get_first_checkpoint_path("unet_pretrained")

model = BinarySegmentator.load_from_checkpoint(provided_ckpt_path, map_location=device)

### Case #2: Use the U-Net model trained in the [preceding notebook (2)](./02_pretrain_unet.ipynb)

In [ ]:
from image_segmentation.models import BinarySegmentator

ckpt_path = dataset_choice.get_first_checkpoint_path("unet_pretraining")

model = BinarySegmentator.load_from_checkpoint(ckpt_path, map_location=device)



### Case #3: Use your own model. In this case you will need to adapt the code below to load your model and its weights, and to use it for generating the binary predictions.

### Case #4: You already have precomputed predictions, or want to use the path classification model on ground truths, you don't need to execute the following cells of this notebook and can directly go to the [path train data generation notebook (4)](./04_path_train_data_generation.ipynb)

## Generating the prediction masks

In [ ]:
from image_segmentation.data.augmentations import build_val_transform
from image_segmentation.data.image_datamodule import ImageDatamodule

datamodule = ImageDatamodule(
    data_dir=data_dir,
    split_file_path=splits_filepath,
    train_split_name=train_split,
    num_workers=0,
    val_batch_size=1,
    shuffle_train=False
)
datamodule.setup()

stats = datamodule.dataset.get_dataset_stats(
    ndim=ndim,
    input_channels=input_channels,
    split_name=train_split,
    split_indices=datamodule.train_indices
)
masked = "foreground" if "foreground" in stats.keys() else "full_image"
mean, std = stats[masked]["mean"], stats[masked]["std"]
val_transforms = build_val_transform(ndim, mean, std)
datamodule.test_transforms = val_transforms
datamodule.setup()

In [ ]:
import gc
import os
import torch
from PIL import Image
from tqdm import tqdm

from image_segmentation.data.io_utils import save_array

dataset = datamodule.dataset
dataset.transforms = val_transforms

pred_dir = os.path.join(data_dir, 'pred')
os.makedirs(pred_dir, exist_ok=True)

model = model.to(device)
model.eval()

with torch.no_grad():
    for i, img_path in enumerate(tqdm(dataset.img_paths)):
        img, _ = dataset[i]
        img = img.unsqueeze(0).float().to(device)

        logits = model._predict(img)
        probs = torch.sigmoid(logits).squeeze().cpu().numpy()

        pred = (probs > 0.5).astype('uint8') * 255

        base_name = os.path.basename(img_path)
        pred_path = os.path.join(pred_dir, base_name)
        
        save_array(pred, pred_path)

        del logits, probs, pred, img

torch.cuda.empty_cache()
gc.collect()

In [ ]:
print(f"Predictions saved to {pred_dir}, total {len(os.listdir(pred_dir))} images.")

Now that we have prediction masks to execute the main model on, you can continue on the [path train data generation notebook (4)](./04_path_train_data_generation.ipynb)